# 10 — Bronze ingest (Event Hubs Capture Avro → Delta)

Reads the previous-hour Avro batch dropped by Event Hubs Capture into ADLS Gen2
and lands the raw payloads into `bronze.transactions`, `bronze.decisions`,
`bronze.cases`. Append-only, partitioned by `ingest_date`.

Trigger: hourly via `orchestrate_medallion.json` Fabric pipeline.

In [ ]:
from datetime import date, datetime, timedelta, timezone

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()

# Pipeline parameter — defaults to the previous hour
ingest_window = spark.conf.get("pipeline.ingest_window", "")
if ingest_window:
    window_start = datetime.fromisoformat(ingest_window)
else:
    window_start = (datetime.now(timezone.utc) - timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)

ingest_date = window_start.date()
print(f"ingest_window={window_start.isoformat()} ingest_date={ingest_date}")

In [ ]:
CAPTURE_ROOT = "abfss://capture@fiprodcap.dfs.core.windows.net"

def capture_path(eventhub: str, ts: datetime) -> str:
    # Default Event Hubs Capture layout: {namespace}/{eventhub}/{partition}/{yyyy}/{MM}/{dd}/{HH}/{mm}/{ss}
    return (
        f"{CAPTURE_ROOT}/fi-prod-ehns/{eventhub}/*/"
        f"{ts.year:04d}/{ts.month:02d}/{ts.day:02d}/{ts.hour:02d}/*/*"
    )

In [ ]:
# ---------------------------------------------------------------------------
# bronze.transactions
# ---------------------------------------------------------------------------

tx_avro = spark.read.format("avro").load(capture_path("tx-events", window_start))

tx_bronze = (
    tx_avro
    .selectExpr("CAST(Body AS STRING) AS payload", "Offset AS eventhub_offset")
    .withColumn("j", F.from_json("payload", """
        tx_id STRING, pan_token STRING, amount DECIMAL(18,2), currency STRING,
        merchant_id STRING, merchant_country STRING, is_card_present BOOLEAN,
        instrument_type STRING, booking_ts TIMESTAMP
    """))
    .select(
        F.col("j.tx_id").alias("tx_id"),
        F.col("j.pan_token").alias("pan_token"),
        F.col("j.amount").alias("amount"),
        F.col("j.currency").alias("currency"),
        F.col("j.merchant_id").alias("merchant_id"),
        F.col("j.merchant_country").alias("merchant_country"),
        F.col("j.is_card_present").alias("is_card_present"),
        F.col("j.instrument_type").alias("instrument_type"),
        F.col("j.booking_ts").alias("booking_ts"),
        F.col("payload").alias("raw_payload"),
        F.lit(ingest_date).cast("date").alias("ingest_date"),
        F.col("eventhub_offset").cast(StringType()),
    )
)

(
    tx_bronze.write.format("delta").mode("append")
    .partitionBy("ingest_date")
    .saveAsTable("lh_fraud.bronze.transactions")
)
print(f"bronze.transactions appended: {tx_bronze.count()} rows")

In [ ]:
# ---------------------------------------------------------------------------
# bronze.decisions
# ---------------------------------------------------------------------------

dec_avro = spark.read.format("avro").load(capture_path("decisions", window_start))

dec_bronze = (
    dec_avro
    .selectExpr("CAST(Body AS STRING) AS payload")
    .withColumn("j", F.from_json("payload", """
        decision_id STRING, tx_id STRING, score DOUBLE, decision STRING,
        sca_outcome STRING, sca_exemption_code STRING, model_version STRING,
        latency_ms INT, decision_ts TIMESTAMP
    """))
    .select(
        F.col("j.*"),
        F.col("payload").alias("raw_payload"),
        F.lit(ingest_date).cast("date").alias("ingest_date"),
    )
)

(
    dec_bronze.write.format("delta").mode("append")
    .partitionBy("ingest_date")
    .saveAsTable("lh_fraud.bronze.decisions")
)
print(f"bronze.decisions appended: {dec_bronze.count()} rows")

In [ ]:
# ---------------------------------------------------------------------------
# bronze.cases  (slower-moving — daily Avro batch from case-mgmt service)
# ---------------------------------------------------------------------------

cases_avro = spark.read.format("avro").load(capture_path("cases", window_start))

cases_bronze = (
    cases_avro
    .selectExpr("CAST(Body AS STRING) AS payload")
    .withColumn("j", F.from_json("payload", """
        case_id STRING, tx_id STRING, fraud_type STRING, confirmed BOOLEAN,
        confirmed_loss_eur DECIMAL(18,2), loss_allocation STRING,
        analyst_id STRING, opened_ts TIMESTAMP, closed_ts TIMESTAMP
    """))
    .select(
        F.col("j.*"),
        F.col("payload").alias("raw_payload"),
        F.lit(ingest_date).cast("date").alias("ingest_date"),
    )
)

(
    cases_bronze.write.format("delta").mode("append")
    .partitionBy("ingest_date")
    .saveAsTable("lh_fraud.bronze.cases")
)
print(f"bronze.cases appended: {cases_bronze.count()} rows")